In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from primer.config import ModelConfig
from primer.model import Transformer


In [3]:
ref_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [4]:
# Qwen3Config {
#   "architectures": [
#     "Qwen3ForCausalLM"
#   ],
#   "attention_bias": false,
#   "attention_dropout": 0.0,
#   "bos_token_id": 151643,
#   "eos_token_id": 151645,
#   "head_dim": 128,
#   "hidden_act": "silu",
#   "hidden_size": 1024,
#   "initializer_range": 0.02,
#   "intermediate_size": 3072,
#   "max_position_embeddings": 40960,
#   "max_window_layers": 28,
#   "model_type": "qwen3",
#   "num_attention_heads": 16,
#   "num_hidden_layers": 28,
#   "num_key_value_heads": 8,
#   "rms_norm_eps": 1e-06,
#   "rope_scaling": null,
#   "rope_theta": 1000000,
#   "sliding_window": null,
#   "tie_word_embeddings": true,
#   "torch_dtype": "float32",
#   "transformers_version": "4.53.1",
#   "use_cache": true,
#   "use_sliding_window": false,
#   "vocab_size": 151936
# }

In [5]:
config = ModelConfig(
    d_model=ref_model.config.hidden_size,
    n_layers=ref_model.config.num_hidden_layers,
    max_seqlen=ref_model.config.max_position_embeddings,
    vocab_size=ref_model.config.vocab_size,
    eos_id=ref_model.config.eos_token_id,
    tie_embeddings=ref_model.config.tie_word_embeddings,
    parallel_layers=False,
    multiple_of=1,
    # ================    
    # Attention Config
    # ================    
    n_heads=ref_model.config.num_attention_heads,
    n_kv_heads=ref_model.config.num_key_value_heads,
    head_dim=ref_model.config.head_dim,
    dropout_p=ref_model.config.attention_dropout,
    attn_bias=ref_model.config.attention_bias,
    attn_prenorm=True,
    attn_postnorm=False,
    qknorm=True,
    qknorm_use_global=False,
    # ==================    
    # FeedForward Config
    # ==================
    intermediate_size=ref_model.config.intermediate_size,
    size_multiplier=None,
    act_fn=ref_model.config.hidden_act,
    gated=True,
    ffw_bias=False,
    ffw_prenorm=True,
    ffw_postnorm=False,
    # ==================
    # Norm Config
    # ==================
    norm_type="RMSNorm",
    norm_eps=ref_model.config.rms_norm_eps,
    # ==================
    # RoPE Config
    # ==================
    rope_theta=ref_model.config.rope_theta,
    rope_pattern="all",
    # ==================
    # Initialization Config
    # ==================
    init_std=ref_model.config.initializer_range,
)
model = Transformer(config)
model

Transformer(
  (tok_embeddings): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x TransformerBlock(
      (attn_prenorm): RMSNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qnorm): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (knorm): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (wq): Linear(in_features=1024, out_features=2048, bias=False)
        (wk): Linear(in_features=1024, out_features=1024, bias=False)
        (wv): Linear(in_features=1024, out_features=1024, bias=False)
        (wo): Linear(in_features=2048, out_features=1024, bias=False)
        (sdpa): ScaledDotProductAttention()
      )
      (ffw_prenorm): RMSNorm((1024,), eps=1e-06, elementwise_affine=True)
      (ffw): FeedForward(
        (act_fn): SiLU()
        (wup): Linear(in_features=1024, out_features=3072, bias=False)
        (wdown): Linear(in_features=3072, out_features=1024, bias=False)
        (wgate): Linear(in_features=1024, out

In [6]:
ref_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [7]:
def load_qwen3_into_transformer(qwen3_model, transformer_model):
    qwen_state_dict = qwen3_model.state_dict()
    transformer_state_dict = transformer_model.state_dict()
    
    new_state_dict = {}
    
    for name, param in qwen_state_dict.items():
        new_name = name

        # --- Top-level ---
        new_name = new_name.replace("model.embed_tokens.", "tok_embeddings.")
        new_name = new_name.replace("model.norm.", "norm.")
        new_name = new_name.replace("lm_head.", "lm_head.")

        # --- Layers ---
        new_name = new_name.replace("model.layers.", "layers.")

        # --- Attention Projections ---
        new_name = new_name.replace(".self_attn.q_proj.", ".attn.wq.")
        new_name = new_name.replace(".self_attn.k_proj.", ".attn.wk.")
        new_name = new_name.replace(".self_attn.v_proj.", ".attn.wv.")
        new_name = new_name.replace(".self_attn.o_proj.", ".attn.wo.")

        # --- Attention Norms ---
        new_name = new_name.replace(".self_attn.q_norm.", ".attn.qnorm.")
        new_name = new_name.replace(".self_attn.k_norm.", ".attn.knorm.")

        # --- Layer Norms ---
        new_name = new_name.replace(".input_layernorm.", ".attn_prenorm.")
        new_name = new_name.replace(".post_attention_layernorm.", ".ffw_prenorm.")

        # --- FeedForward (MLP) ---
        new_name = new_name.replace(".mlp.gate_proj.", ".ffw.wgate.")
        new_name = new_name.replace(".mlp.up_proj.", ".ffw.wup.")
        new_name = new_name.replace(".mlp.down_proj.", ".ffw.wdown.")

        # Skip rotary embedding
        if "rotary_emb" in name:
            continue

        if new_name in transformer_state_dict:
            if transformer_state_dict[new_name].shape != param.shape:
                print(f"[shape mismatch] {new_name}: {param.shape} vs {transformer_state_dict[new_name].shape}")
                continue
            new_state_dict[new_name] = param
        else:
            print(f"[skipped] No match for: {name} → {new_name}")

    # Load into model
    missing_keys, unexpected_keys = transformer_model.load_state_dict(new_state_dict, strict=False)

    print("\n✅ Loaded parameters.")
    print(f"\t🔸 Missing keys in Transformer: {missing_keys}")
    print(f"\t🔸 Unexpected keys from Qwen3: {unexpected_keys}")

    return transformer_model


In [8]:
model = load_qwen3_into_transformer(ref_model, model)


✅ Loaded parameters.
	🔸 Missing keys in Transformer: ['freqs_cis']
	🔸 Unexpected keys from Qwen3: []


In [9]:
input_ids = torch.randint(0, config.vocab_size, (2, 10))  # Example input

In [12]:
ref_model = ref_model.eval()
model = model.eval()

In [13]:
with torch.inference_mode():
    ref_out = ref_model(input_ids=input_ids).logits
ref_out

tensor([[[ 4.5913,  4.5549,  5.4085,  ...,  1.7557,  1.7557,  1.7557],
         [ 6.9070,  6.4219,  5.2501,  ..., -1.1982, -1.1982, -1.1982],
         [ 6.5444,  7.5264,  3.2035,  ..., -0.9344, -0.9344, -0.9344],
         ...,
         [ 8.3704,  7.9936,  9.0875,  ..., -0.3761, -0.3761, -0.3761],
         [ 8.6945,  6.4051,  9.0913,  ..., -1.4981, -1.4981, -1.4981],
         [ 5.0154,  5.3357,  4.4159,  ...,  1.0189,  1.0189,  1.0189]],

        [[ 4.8235,  5.0330,  5.1756,  ...,  2.0042,  2.0042,  2.0042],
         [ 4.3364,  5.0385,  6.9565,  ...,  0.4753,  0.4753,  0.4753],
         [ 2.9456,  6.0892,  6.5876,  ...,  1.1334,  1.1334,  1.1334],
         ...,
         [ 6.0721,  3.0865,  3.6538,  ..., -0.4289, -0.4289, -0.4289],
         [ 4.1619,  2.6951,  3.7570,  ...,  1.0599,  1.0599,  1.0599],
         [ 4.4624,  2.6843,  5.7153,  ...,  0.5126,  0.5126,  0.5126]]])

In [14]:
with torch.inference_mode():
    out = model(input_ids=input_ids)
out

tensor([[[ 4.5913,  4.5549,  5.4085,  ...,  1.7557,  1.7557,  1.7557],
         [ 6.4293,  6.6585,  5.0534,  ..., -1.2540, -1.2540, -1.2540],
         [ 4.2255,  5.9250,  3.8822,  ..., -1.9724, -1.9724, -1.9724],
         ...,
         [ 8.0456,  7.0833,  7.4397,  ..., -0.3518, -0.3518, -0.3518],
         [ 8.5954,  5.9944,  7.8990,  ..., -1.2915, -1.2915, -1.2915],
         [ 4.5640,  5.3861,  4.7818,  ...,  0.6622,  0.6622,  0.6622]],

        [[ 4.8235,  5.0330,  5.1756,  ...,  2.0042,  2.0042,  2.0042],
         [ 5.4212,  5.3979,  7.3100,  ...,  0.6155,  0.6155,  0.6155],
         [ 3.0654,  7.6872,  8.1054,  ...,  0.3822,  0.3822,  0.3822],
         ...,
         [ 5.5549,  2.7219,  3.3454,  ...,  0.0353,  0.0353,  0.0353],
         [ 4.4067,  2.6971,  3.6364,  ...,  0.8982,  0.8982,  0.8982],
         [ 5.2893,  3.4221,  6.1206,  ...,  0.4983,  0.4983,  0.4983]]])

In [15]:
with torch.inference_mode():
    ref_emb = ref_model.model.embed_tokens(input_ids) 
    emb = model.tok_embeddings(input_ids)
torch.allclose(ref_emb, emb, atol=1e-5)  # Check if embeddings match

True

In [67]:
from transformers.models.qwen3.modeling_qwen3 import apply_rotary_pos_emb

with torch.inference_mode():
    pos_ids = torch.arange(input_ids.shape[1]).unsqueeze(0)  # Create a tensor for positions
    pos_emb = ref_model.model.rotary_emb(ref_emb, pos_ids)
    ref_att = ref_model.model.layers[0].self_attn
    
    hidden_states = ref_model.model.layers[0].input_layernorm(ref_emb)

    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, ref_model.config.head_dim)

    query_states = ref_att.q_norm(ref_att.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
    key_states = ref_att.k_norm(ref_att.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
    value_states = ref_att.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

    cos, sin = pos_emb
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    # ref_h = ref_model.model.layers[0].self_attn(ref_h_norm, attention_mask=None, position_embeddings=pos_emb)[0]
    # ref_out = ref_model.model.layers[0](ref_emb, position_embeddings=pos_emb)[0]

In [68]:
query_states.shape, key_states.shape, value_states.shape

(torch.Size([2, 16, 10, 128]),
 torch.Size([2, 8, 10, 128]),
 torch.Size([2, 8, 10, 128]))

In [47]:
cos, sin = pos_emb
freqs_cis = torch.complex(cos[0], sin[0]) 

In [48]:
freqs_cis.shape, model.freqs_cis.shape

(torch.Size([10, 128]), torch.Size([40960, 64]))

In [41]:
with torch.inference_mode():
    freqs_cis = model.freqs_cis
    x = model.layers[0].attn_prenorm(ref_emb)
    layer = model.layers[0].attn
    
    bs, seqlen, _ = x.shape
    xq = layer.wq(x)  # (bs, seqlen, n_local_heads * head_dim)
    xk = layer.wk(x)  # (bs, seqlen, n_kv_heads * head_dim)
    xv = layer.wv(x)  # (bs, seqlen, n_kv_heads * head_dim)

    # ==== QK Normalization if applicable
    if layer.use_qknorm and layer.qknorm_use_global:
        # (Option 1) Normalize over the full concatenated dimension first
        xq = layer.qnorm(xq)  # no shape change
        xk = layer.knorm(xk)  # no shape change

    # NOTE: using -1 instead of `n_heads` (or `n_kv_heads`) to infer the actual local heads
    # from sizes of xq, xk, and xv as TP may have sharded them after the above linear ops
    xq = xq.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_local_heads, head_dim)
    xk = xk.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_kv_heads, head_dim)
    xv = xv.view(bs, seqlen, -1, layer.head_dim)  # (bs, seqlen, n_kv_heads, head_dim)

    if layer.use_qknorm and not layer.qknorm_use_global:
        # (Option 2) Normalise each head independently after reshaping
        xq = layer.qnorm(xq)  # no shape change
        xk = layer.knorm(xk)  # no shape change
    
    # h = model.layers[0].attn(h_norm, freqs_cis)

In [69]:
torch.allclose(hidden_states, x)

True

In [87]:
def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Reshape frequency tensor to be broadcast-compatible with x.
    x is (bs, seqlen, n_heads, head_dim) or (bs, seqlen, n_kv_heads, head_dim)
    freqs_cis is (seqlen, head_dim // 2) or (bs, seqlen, head_dim)
    """
    seqlen = x.shape[1]
    head_dim_half = x.shape[-1] // 2

    # Handle unbatched freqs_cis (pre-computation)
    if freqs_cis.dim() == 2:
        freqs_cis = freqs_cis[:seqlen, :head_dim_half]
        # want (1, seqlen, 1, head_dim // 2)
        return freqs_cis.unsqueeze(0).unsqueeze(2)
    # Handle batched freqs_cis (from reference model)
    else:
        # freqs_cis is (bs, seqlen, head_dim), needs to be (bs, seqlen, 1, head_dim // 2)
        # Reshape to view as complex numbers of size head_dim//2
        freqs_cis_complex = torch.view_as_complex(freqs_cis.float().reshape(*freqs_cis.shape[:-1], -1, 2))
        return freqs_cis_complex.unsqueeze(2)

def apply_rotary_emb(xq: torch.Tensor, xk: torch.Tensor, freqs_cis: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Apply rotary embeddings to input tensors using the given frequency tensor.
    """
    # Reshape to (bs, seqlen, n_heads, head_dim)
    xq_ = xq.float().reshape(*xq.shape[:-1], -1, 2)
    xk_ = xk.float().reshape(*xk.shape[:-1], -1, 2)

    # Reshape freqs_cis for broadcasting
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)

    # Convert to complex numbers
    xq_complex = torch.view_as_complex(xq_)
    xk_complex = torch.view_as_complex(xk_)

    # Apply rotary embeddings
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(3)

    return xq_out.type_as(xq), xk_out.type_as(xk)

In [88]:
with torch.inference_mode():
    q, k = apply_rotary_emb(xq, xk, freqs_cis)
    q = q.transpose(1, 2)
    k = k.transpose(1, 2)

In [89]:
torch.allclose(q, query_states, atol=1e-5)

False

In [90]:
q.shape, query_states.shape, k.shape, key_states.shape

(torch.Size([2, 16, 10, 128]),
 torch.Size([2, 16, 10, 128]),
 torch.Size([2, 8, 10, 128]),
 torch.Size([2, 8, 10, 128]))

In [91]:
torch.allclose(q, query_states, atol=1e-5), torch.allclose(k, key_states, atol=1e-5)

(False, False)

In [92]:
q == query_states

tensor([[[[ True,  True,  True,  ...,  True,  True,  True],
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False],
          ...,
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False]],

         [[ True,  True,  True,  ...,  True,  True,  True],
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False],
          ...,
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False]],

         [[ True,  True,  True,  ...,  True,  True,  True],
          [False, False, False,  ..., False, False, False],
          [False, False, False,  ..., False, False, False],
          ...,
          [False, False, False,  ..., False, False,

In [ ]:

ref_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwe